In [7]:
import os
import re
import json
import urllib3
from time import sleep
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

urllib3.disable_warnings()

BASE_URL    = "https://sinaica.inecc.gob.mx/pags/datGrafs.php"
STATION_URL = "https://sinaica.inecc.gob.mx/pags/datosRed.php"

START       = pd.Timestamp("2022-01-01")
END         = pd.Timestamp("2025-12-31")
OUT_DIR     = "data/sinaica/"
MAX_WORKERS = 16
os.makedirs(OUT_DIR, exist_ok=True)
# ppm → target unit; PM already in µg/m³
PARAM_INFO = {
    "PM2.5": {"col": "PM2.5 (µg/m³)",  "suffix": "PM2.5", "factor": 1.0},
    "PM10":  {"col": "PM10 (µg/m³)",   "suffix": "PM10",  "factor": 1.0},
    "O3":    {"col": "Ozone (µg/m³)",  "suffix": "Ozone", "factor": 1.96 * 1000},
    "NO2":   {"col": "NO2 (µg/m³)",    "suffix": "NO2",   "factor": 1.88 * 1000},
    "SO2":   {"col": "SO2 (µg/m³)",    "suffix": "SO2",   "factor": 2.62 * 1000},
    "CO":    {"col": "CO (mg/m³)",     "suffix": "CO",    "factor": 1.15},
}
PARAMS = list(PARAM_INFO.keys())


In [8]:
def get_stations() -> pd.DataFrame:
    r = requests.get(STATION_URL, verify=False, timeout=30)
    soup = BeautifulSoup(r.text, "html.parser")
    select = soup.find("select", {"id": "selPickHeadEst"})
    return pd.DataFrame([
        {
            "station_id":   opt["value"],
            "station_name": opt.text.strip(),
            "raw_tokens":   opt.get("data-tokens", ""),
        }
        for opt in select.find_all("option")
        if opt.get("value", "").strip()
    ])


def get_year(session, station_id, parameter, year) -> pd.DataFrame:
    payload = {
        "estacionId": station_id,
        "param":      parameter,
        "fechaIni":   f"{year}-01-01",
        "rango":      f"{year}-12-31",  # end date, not number of days
        "tipoDatos":  "",
    }
    r = session.post(
        BASE_URL, data=payload,
        headers={"Referer": "https://sinaica.inecc.gob.mx/"},
        verify=False, timeout=30,
    )
    r.raise_for_status()

    match = re.search(r"var dat\s*=\s*(\[.*?\]);", r.text, re.DOTALL)
    if not match:
        return pd.DataFrame()
    data = json.loads(match.group(1))
    if not data:
        return pd.DataFrame()

    df = pd.DataFrame(data)
    # rename by key name, not position, to be robust against API key order changes
    df = df.rename(columns={"id": "row_id", "fecha": "date", "hora": "hour",
                             "valor": "value", "bandO": "extra", "val": "valid"})
    df["datetime"] = (
        pd.to_datetime(df["date"])
        + pd.to_timedelta(pd.to_numeric(df["hour"], errors="coerce"), unit="h")
    )
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["valid"] = pd.to_numeric(df["valid"], errors="coerce")
    return df[["datetime", "value", "valid"]]


def scrape_station(station_id, station_name) -> str:
    target_index = pd.date_range(start=START, end=END, freq="h")
    session      = requests.Session()
    saved        = 0
    skipped      = 0

    for param, info in PARAM_INFO.items():
        out_path = os.path.join(OUT_DIR, f"station_{station_id}_{info['suffix']}.csv")
        if os.path.exists(out_path):
            skipped += 1
            continue

        yearly = []
        for year in range(START.year, END.year + 1):
            try:
                df = get_year(session, station_id, param, year)
                if not df.empty:
                    yearly.append(df)
            except Exception:
                pass
            sleep(0.3)

        if not yearly:
            continue

        combined = pd.concat(yearly, ignore_index=True)
        combined = combined.dropna(subset=["value"])
        combined = combined[combined["valid"] == 1]
        combined = combined.drop_duplicates(subset="datetime")
        combined = combined.set_index("datetime").sort_index()

        # convert units and rename column
        combined["value"] = combined["value"] * info["factor"]
        combined = combined[["value"]].rename(columns={"value": info["col"]})

        # reindex to full hourly range (NaN for missing hours)
        combined = combined.reindex(target_index)
        combined.index.name = "Timestamp"

        combined.to_csv(out_path)
        saved += 1

    if skipped == len(PARAM_INFO):
        return f"skip:{station_id}"
    if saved == 0:
        return f"empty:{station_id}"
    return f"save:{station_id}:{saved} new, {skipped} skipped"


In [9]:
# ── Fetch & save station list ─────────────────────────────────────────────────
df_stations = get_stations()
df_stations.to_csv(os.path.join(OUT_DIR, "sinaica_stations.csv"), index=False)
print(f"Found {len(df_stations)} stations")
df_stations.head()


Found 181 stations


,station_id,station_name,raw_tokens
0,356,Presidencia Municipal,Guanajuato GTO ABA Abasolo Presidencia Municip...
1,31,CBTIS,Aguascalientes AGS AGS Aguascalientes CBTIS CBT
2,33,Centro,Aguascalientes AGS AGS Aguascalientes Centro CEN
3,303,Instituto Educativo,Aguascalientes AGS AGS Aguascalientes Institut...
4,32,Secretaría de Medio Ambiente,Aguascalientes AGS AGS Aguascalientes Secretar...


In [30]:
# ── Scrape all stations (threaded) ───────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
saved, skipped, empty = 0, 0, 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(scrape_station, row["station_id"], row["station_name"]): row["station_id"]
        for _, row in df_stations.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Stations"):
        try:
            result = future.result()
            if result.startswith("save"):
                saved += 1
            elif result.startswith("skip"):
                skipped += 1
            else:
                empty += 1
        except Exception as e:
            print(f"Error on station {futures[future]}: {e}")
            empty += 1

print(f"\nDone. Saved: {saved} | Skipped (existing): {skipped} | No data: {empty}")


Stations:   0%|          | 0/181 [00:00<?, ?it/s]


Done. Saved: 74 | Skipped (existing): 107 | No data: 0


In [10]:

# ── Probe earliest year with data per param (10 random stations) ──────────────
AVAIL_DIR   = "data/sinaica_availability/"
PROBE_START = 2000
PROBE_END   = 2022

os.makedirs(AVAIL_DIR, exist_ok=True)

sample_stations = df_stations.sample(10, random_state=42)


def probe_station_param(station_id, param):
    """Return (station_id, param, first_year) or first_year=None if no data found."""
    session = requests.Session()
    for year in range(PROBE_START, PROBE_END + 1):
        try:
            df = get_year(session, station_id, param, year)
            if not df.empty:
                return station_id, param, year
        except Exception:
            pass
        sleep(0.2)
    return station_id, param, None


combos = [
    (row["station_id"], param)
    for _, row in sample_stations.iterrows()
    for param in PARAMS
]

probe_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(probe_station_param, sid, param): (sid, param) for sid, param in combos}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Probing availability"):
        try:
            sid, param, first_year = future.result()
            probe_results.append({"station_id": sid, "parameter": param, "first_year": first_year})
        except Exception as e:
            sid, param = futures[future]
            print(f"Error {sid}/{param}: {e}")

df_avail = pd.DataFrame(probe_results)
avail_path = os.path.join(AVAIL_DIR, "data_availability.csv")
df_avail.to_csv(avail_path, index=False)
print(f"Saved → {avail_path}")

df_avail.pivot(index="station_id", columns="parameter", values="first_year")


Probing availability:   0%|          | 0/60 [00:00<?, ?it/s]

Saved → data/sinaica_availability/data_availability.csv


parameter,CO,NO2,O3,PM10,PM2.5,SO2
station_id,,,,,,
186,NaN,NaN,NaN,NaN,NaN,NaN
244,2012.0,2000.0,2011.0,2011.0,2003.0,2011.0
251,2012.0,2000.0,2012.0,2012.0,2012.0,2012.0
388,2008.0,2000.0,2016.0,2014.0,2015.0,2013.0
427,2019.0,2000.0,2020.0,NaN,NaN,2020.0
44,2019.0,NaN,2019.0,NaN,NaN,2019.0
479,NaN,NaN,NaN,NaN,NaN,NaN
53,2012.0,2000.0,2012.0,2012.0,2016.0,2012.0
57,2013.0,2013.0,2013.0,2013.0,2013.0,2013.0
